# Streaming Response With In-Text Citations

**Author:** Shinin Varongchayakul

**Date:** 23 Jul 2026

## 1. Get Retrieved Documents

In [1]:
# Set retrieved documents
retrieved_docs = [
    {
        "id": "hr_0001",
        "content": "Employees are entitled to 12 days of annual leave after completing one year of service.",
        "metadata": {
            "source": "Employee Handbook.pdf",
            "author": "HR Department",
            "date": "2025-01-15"
        }
    },
    {
        "id": "hr_0002",
        "content": "Employees may work remotely up to two days per week with manager approval.",
        "metadata": {
            "source": "Remote Work Policy.pdf",
            "author": "HR Department",
            "date": "2025-03-10"
        }
    },
    {
        "id": "hr_0003",
        "content": "Travel expenses must be submitted within 30 days after the business trip.",
        "metadata": {
            "source": "Expense Policy.pdf",
            "author": "Finance Department",
            "date": "2024-11-01"
        }
    }
]

## 2. Set Response Generator

### 2.1 Prompt

In [2]:
# Set system prompt
system_prompt = """
You are an AI assistant that answers user questions using ONLY the documents provided below.

Cite every claim inline using the document's ID exactly as given, wrapped in square brackets (e.g., [hr_0010]).

Only cite IDs you actually use to answer the question. Do not invent IDs.
"""

# Set user prompt
user_prompt = """
Documents:
{docs}

User's query:
{query}
"""

In [3]:
# Import package
from langchain_core.prompts import ChatPromptTemplate

# Set prompt template
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt.strip()),
        ("user", user_prompt.strip())
    ]
)

### 2.2 LLM

In [4]:
# Import packages
from pathlib import Path
from dotenv import load_dotenv
import os

# Get .env file path
PROJECT_ROOT = Path.cwd().parents[2]
env_path = PROJECT_ROOT / ".env"

# Load variables from .env
load_dotenv(env_path, override=True)

# Get Gemini API key
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY_01")

In [5]:
# Import package
from langchain_google_genai import ChatGoogleGenerativeAI

# Create model instance
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2,
    api_key=GEMINI_API_KEY
)

### 2.3 Chain Pipeline

In [6]:
# Import package
from langchain_core.output_parsers import StrOutputParser

# Create chain pipeline
chain = prompt_template | llm | StrOutputParser()

# 3. Create Functions

### 3.1 Document Formatter

In [7]:
# Format documents
def format_docs(docs: list[dict]) -> str:

    # Instantiate collector
    formatted = []

    # Loop through documents
    for doc in docs:
        metadata = doc.get("metadata", {})
        formatted.append(
            f"ID: {doc['id']}\n"
            f"Content: {doc['content']}\n"
            f"Source: {metadata.get('source', 'N/A')}\n"
            "-------------------------"
        )

    # Return formatted string
    return "\n".join(formatted)

In [8]:
# Import package
import re

# Define function to stream response with citations
def stream_response_with_citations(query: str, docs: list[dict]) -> None:

    # Set regex pattern for citations
    CITE_RE = re.compile(r"\[([A-Za-z0-9_\-]+)\]")

    # Build lookup dicts
    docs_by_id = {
        doc["id"]: doc for doc in docs
    }

    # Set required variables
    seen_ids: set[str] = set()
    ordered_citations: list[dict] = []
    buffer = ""

    # Loop through streaming
    for chunk in chain.stream(
        {
            "docs": format_docs(docs),
            "query": query
        }
    ):
        
        # Add chunk to buffer
        buffer += chunk

        # Yield response text
        yield {
            "type": "token",
            "content": chunk
        }

        # Re-scan full buffer
        for match in CITE_RE.finditer(buffer):
            doc_id = match.group(1)
            if doc_id not in seen_ids and doc_id in docs_by_id:
                seen_ids.add(doc_id)
                ordered_citations.append(docs_by_id[doc_id])
    
    # Yield citations
    yield {
        "type": "citation",
        "content": ordered_citations
    }

In [ ]:
# Define function to print streaming generator
def print_streamed_response(event_stream) -> list[dict]:

    # Set default if no event is yielded
    ordered_citations: list[dict] = []

    # Loop through yielded events
    for event in event_stream:

        # Check if event is response text
        if event["type"] == "token":
            print(
                event["content"],
                end="",
                flush=True
            )

        # Check if event is citations
        elif event["type"] == "citation":
            ordered_citations = event["content"]
            print("\n\n--- Sources ---")

            for c in ordered_citations:
                print(f"[{c['id']}] {c['metadata']['source']}")

    # Return citations
    return ordered_citations

## 4. Generate Response

In [10]:
# Set user query
user_query = "How many annual leave days do employees receive, and can they work remotely?"

# Generate response
citations = print_streamed_response(
    stream_response_with_citations(
        query=user_query,
        docs=retrieved_docs
    )
)

Employees are entitled to 12 days of annual leave after completing one year of service [hr_0001]. They may work remotely up to two days per week with manager approval [hr_0002].

--- Citations (in order of first appearance) ---
[hr_0001] Employee Handbook.pdf
[hr_0002] Remote Work Policy.pdf
